In [1]:
### arg pipelines

In [3]:
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path



In [4]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: artificial_intelligence_intro.pdf
✓ Loaded 1 pages

Processing: data_science_overview.pdf
✓ Loaded 1 pages

Processing: machine_learning_basics.pdf
✓ Loaded 1 pages

Processing: python_intro.pdf
✓ Loaded 1 pages

Total documents loaded: 4


In [5]:
all_pdf_documents

[Document(metadata={'source': '..\\data\\pdf_files\\artificial_intelligence_intro.pdf', 'page': 0, 'source_file': 'artificial_intelligence_intro.pdf', 'file_type': 'pdf'}, page_content='Artificial Intelligence Introduction\nArtificial Intelligence focuses on building systems that can perform tasks requiring human\nintelligence.\nExamples include speech recognition, image classification, recommendation systems, and\nchatbots.\nAI includes subfields such as Machine Learning, Deep Learning, and Natural Language\nProcessing.\n'),
 Document(metadata={'source': '..\\data\\pdf_files\\data_science_overview.pdf', 'page': 0, 'source_file': 'data_science_overview.pdf', 'file_type': 'pdf'}, page_content='Data Science Overview\nData Science combines statistics, programming, and domain knowledge to extract insights from\ndata.\nIt involves data collection, cleaning, analysis, visualization, and predictive modeling.\nPython and R are the most commonly used programming languages in this field.\n'),
 D

In [6]:
### splitting documents

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)

Split 4 documents into 4 chunks

Example chunk:
Content: Artificial Intelligence Introduction
Artificial Intelligence focuses on building systems that can perform tasks requiring human
intelligence.
Examples include speech recognition, image classification,...
Metadata: {'source': '..\\data\\pdf_files\\artificial_intelligence_intro.pdf', 'page': 0, 'source_file': 'artificial_intelligence_intro.pdf', 'file_type': 'pdf'}


In [8]:
chunks

[Document(metadata={'source': '..\\data\\pdf_files\\artificial_intelligence_intro.pdf', 'page': 0, 'source_file': 'artificial_intelligence_intro.pdf', 'file_type': 'pdf'}, page_content='Artificial Intelligence Introduction\nArtificial Intelligence focuses on building systems that can perform tasks requiring human\nintelligence.\nExamples include speech recognition, image classification, recommendation systems, and\nchatbots.\nAI includes subfields such as Machine Learning, Deep Learning, and Natural Language\nProcessing.'),
 Document(metadata={'source': '..\\data\\pdf_files\\data_science_overview.pdf', 'page': 0, 'source_file': 'data_science_overview.pdf', 'file_type': 'pdf'}, page_content='Data Science Overview\nData Science combines statistics, programming, and domain knowledge to extract insights from\ndata.\nIt involves data collection, cleaning, analysis, visualization, and predictive modeling.\nPython and R are the most commonly used programming languages in this field.'),
 Docum

In [9]:
###embedding and vector stoe db

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\DELL\Desktop\CropAI\virtualEN\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

print(embedding_manager)

Loading embedding model: all-MiniLM-L6-v2


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Model loaded successfully. Embedding dimension: 384


In [ ]:
import faiss
import numpy as np
import pickle
from typing import List, Any


class FAISSVectorStore:
    """Simple FAISS vector store for document retrieval"""

    def __init__(self, embedding_dim: int = 384, index_path: str = "../data/faiss_index"):
        self.embedding_dim = embedding_dim
        self.index_path = index_path

        self.index = faiss.IndexFlatL2(embedding_dim)
        self.documents = []

        print("FAISS Vector Store initialized")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and embeddings to FAISS"""

        print(f"Adding {len(documents)} documents to FAISS...")

        embeddings = np.array(embeddings).astype("float32")
        self.index.add(embeddings)

        for doc in documents:
            self.documents.append(doc)

        print(f"Total vectors stored: {self.index.ntotal}")

    def similarity_search(self, query_embedding: np.ndarray, k: int = 3):
        """Search similar documents.

        Returns:
            List of (document, distance) tuples sorted by distance (ascending).
        """
        # FAISS requires a 2D float32 array as input
        query_embedding = np.array(query_embedding).reshape(1, -1).astype("float32")

        distances, indices = self.index.search(query_embedding, k)

        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx != -1:  # FAISS returns -1 when there are fewer results than k
                results.append((self.documents[idx], float(dist)))

        return results

    def save(self):
        """Save index and documents"""

        faiss.write_index(self.index, f"{self.index_path}.index")

        with open(f"{self.index_path}.pkl", "wb") as f:
            pickle.dump(self.documents, f)

        print("FAISS index saved")

    def load(self):
        """Load index and documents"""

        self.index = faiss.read_index(f"{self.index_path}.index")

        with open(f"{self.index_path}.pkl", "rb") as f:
            self.documents = pickle.load(f)

        print("FAISS index loaded")


In [17]:
vectorstore = FAISSVectorStore()

FAISS Vector Store initialized


In [19]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings) 


Generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.91it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to FAISS...
Total vectors stored: 4


In [20]:
vectorstore.save()

FAISS index saved


In [ ]:
class RAGRetriever:
    """Retriever for RAG using FAISS"""

    def __init__(self, vector_store: FAISSVectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever

        Args:
            vector_store: FAISS vector store instance
            embedding_manager: embedding manager instance
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents for a query

        Args:
            query: The input query string
            top_k: Number of top results to return
            score_threshold: Minimum similarity score (0-1) to include a result

        Returns:
            List of retrieved documents with metadata and scores
        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # Returns list of (document, l2_distance) tuples
            results = self.vector_store.similarity_search(
                query_embedding=query_embedding,
                k=top_k,
            )

            retrieved_docs = []

            for rank, (doc, distance) in enumerate(results):
                # Convert L2 distance to a 0-1 similarity score
                score = 1 / (1 + distance)

                if score >= score_threshold:
                    retrieved_docs.append({
                        "content": doc.page_content,
                        "metadata": doc.metadata,
                        "score": round(score, 4),
                        "distance": round(distance, 4),
                        "rank": rank + 1,
                    })

            print(f"Retrieved {len(retrieved_docs)} documents.")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [39]:
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

In [40]:
rag_retriever

In [42]:
rag_retriever.retrieve("What are the main topics covered in the PDF documents?")

Retrieving documents for query: 'What are the main topics covered in the PDF documents?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.91it/s]

Generated embeddings with shape: (1, 384)
Error during retrieval: similarity_search() got an unexpected keyword argument 'query'


[]